# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All explorations use entity `@id` references for full interoperability and schema compliance.

### Dataset Source
The dataset Croissant schema source is:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect its summary using `mlcroissant`. All access will use entity `@id`s as per Croissant best practices.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata
print('Dataset Title:', dataset.metadata.name)
print('\nDescription:')
print(dataset.metadata.description)
print('\nLicense:', dataset.metadata.license)
print('Published:', getattr(dataset.metadata, 'datePublished', 'N/A'))
print('Identifier:', getattr(dataset.metadata, 'identifier', 'N/A'))
print('Keywords:', getattr(dataset.metadata, 'keywords', 'N/A'))

## 2. Data Overview
Review core record sets, their `@id`s, available fields, and columns (`@id`s only) as defined by the Croissant schema. All relationships and data access are shown by `@id`.

In [ ]:
# List all available record sets by `@id`
print('Available Record Sets (@id):')
for rs in dataset.record_sets:
    print('  -', rs.id)

# For each record set, print its fields and columns by @id
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    if rs.fields:
        print('  Fields:')
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
            if hasattr(field, 'column') and field.column is not None:
                print(f"      Column @id: {getattr(field.column, 'id', str(field.column))}")
    else:
        print('  No fields defined.')

# For programmatic usage, build a list of all record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]

## 3. Data Extraction
Extract records from selected record sets (referenced by their `@id`) and load into pandas DataFrames for further analysis. DataFrame column names are the field `@id`s.

In [ ]:
# Extract all record sets found by their @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded record set: {rs_id} with shape {dataframes[rs_id].shape}")
    else:
        print(f"No records found for record set: {rs_id}")

# Display columns (field @ids) of the first non-empty DataFrame
first_nonempty_rs = next((rs_id for rs_id, df in dataframes.items() if not df.empty), None)
if first_nonempty_rs:
    print(f"\nColumns in {first_nonempty_rs}:\n", dataframes[first_nonempty_rs].columns.tolist())
    dataframes[first_nonempty_rs].head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (by its `@id`) from the extracted data and demonstrate typical filtering, normalization, and grouping steps. All field accesses are by `@id`. Please replace example IDs with ones from your specific record set as required.

In [ ]:
# Example usage: replace these IDs based on above overview!
selected_record_set_id = first_nonempty_rs  # Choose a valid record set ID
df = dataframes[selected_record_set_id]

# List all columns for manual inspection
print('Columns available (field @id):', df.columns.tolist())

# Suppose the dataset has a numeric field with @id 'coefficient' and a group field 'variable'.
# Replace these with the actual @ids.
numeric_field_id = None
group_field_id = None
# Try to auto-detect a likely numeric field and group field
for col in df.columns:
    if 'coefficient' in col.lower() or 'value' in col.lower() or 'loglikelihood' in col.lower():
        numeric_field_id = col
    if 'variable' in col.lower() or 'group' in col.lower() or 'class' in col.lower():
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = df.columns[0]  # Fallback

print(f"\nUsing numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Grouping by: {group_field_id}")

# Filtering: select rows where value > threshold (e.g., > 0 if possible)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].quantile(0.8) if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records where {numeric_field_id} > {threshold:.3f} (top 5):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping if possible
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field (by `@id`) and, if possible, plot its mean by group (using the group field's `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=30)
plt.title(f"Distribution of field (@id): {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping is available, plot mean per group
if group_field_id and group_field_id in df.columns:
    means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    plt.figure(figsize=(10,5))
    means.plot(kind='bar')
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded dataset metadata and records via the Croissant schema using `mlcroissant`;
- Accessed record sets, fields, and columns using their stable `@id` identifiers;
- Extracted and explored raw data in pandas DataFrames;
- Performed simple EDA steps, including filtering, normalizing, and grouping fields using only `@id` references;
- Visualized distributions and relationships in the dataset.

This approach—referencing all entities by `@id`—ensures your analyses are reproducible and compliant with the Croissant specification. For further analysis, consult the data dictionary or schema and use the appropriate `@id`s for custom exploration.